# Google ADK Fundamentals

**Notebook 1 of 2 — learn the building blocks**

Follow Helen Parr from a general insurance question to a policy lookup and a specialist handoff. Each example adds one concept. [Notebook 2](02_Insurance_Claims_Agent.ipynb) combines them into a complete claims workflow.

## What you will learn

1. Give a model a job with agent instructions.
2. Connect an agent to facts through a tool.
3. Use several tools within one agent.
4. Carry facts between messages with session state.
5. Route a request to a specialist.
6. Run required stages in order with a workflow.

```text
Model → Agent → Tool → State → Specialists → Workflow
```

![ADK Fundamentals: The Big Picture](Images/01_adk_fundamentals_big_picture.png)

## 0. Set up the notebook

Run `uv sync`, select the project's `.venv` kernel, and run cells from top to bottom with the project folder as the working directory.

- Add `GEMINI_API_KEY` or `GOOGLE_API_KEY` to `.env` for live agent examples.
- Without a key, or with `RUN_LLM_EXAMPLES=false`, direct tool examples still run.
- Set `INTERACTIVE_INPUTS=true` to type your own messages in the two-turn state example.

The examples read policy and pricing data from `local_data`.

In [1]:
import importlib.metadata
import json
import os
from html import escape
from pathlib import Path
from pprint import pformat

from dotenv import load_dotenv
from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types
from IPython.display import HTML, display
import logging

logging.getLogger("google_genai.models").setLevel(logging.ERROR)

load_dotenv()

PROJECT_ROOT = Path.cwd()
MODEL = os.getenv("ADK_MODEL", "gemini-2.5-flash")
HAS_MODEL_ACCESS = bool(os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"))
RUN_LLM_EXAMPLES = HAS_MODEL_ACCESS and os.getenv("RUN_LLM_EXAMPLES", "true").lower() == "true"
INTERACTIVE_INPUTS = os.getenv("INTERACTIVE_INPUTS", "false").lower() == "true"

POLICIES = json.loads((PROJECT_ROOT / "local_data" / "policies.json").read_text())
PARTS_PRICING = json.loads((PROJECT_ROOT / "local_data" / "parts_pricing.json").read_text())

print("google-adk:", importlib.metadata.version("google-adk"))
print("model:", MODEL)
print("live model examples:", RUN_LLM_EXAMPLES)
print("policies loaded from:", PROJECT_ROOT / "local_data" / "policies.json")

google-adk: 2.8.0
model: gemini-3.6-flash
live model examples: True
policies loaded from: ./local_data/policies.json


## 1. Give the model a job

A **model** generates a response from the context it receives. An **agent** gives it a job, instructions, and access to tools:

```text
Agent = Model + Instructions + Optional tools
```

The first agent explains general insurance terms. Its instructions define its role and limits; it has no tool to look up Helen's policy.

In [2]:
# General agent: explain insurance terms using instructions alone.
general_claims_agent = Agent(
    name="general_claims_agent",
    model=MODEL,
    description="Explains general auto-insurance concepts to a beginner.",
    instruction="""
You are a patient auto-insurance educator.
- Explain general concepts in plain language and use one small example when helpful.
- Keep the answer under 150 words unless the user asks for more detail.
- You do not have access to private policy records in this role.
- If asked about a specific policy, say that a policy lookup is required.
- Never invent coverage, claim, customer, or payment facts.
""",
)

print("agent:", general_claims_agent.name)
print("tools:", general_claims_agent.tools)

agent: general_claims_agent
tools: []


### Helpers: run an agent and read its trace

A **runner** executes the agent; a **session** holds the conversation and its state. These helpers send messages and display the returned **events**, which record what happened.

Run this helper cell once, then follow the examples: **message → agent activity → reply**.

In [3]:
# Runner helper: create an agent runner and a fresh in-memory session.
async def start_agent_session(agent, app_name: str, initial_state: dict | None = None):
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = await runner.session_service.create_session(
        app_name=app_name,
        user_id="workshop-user",
        state=initial_state or {},
    )
    return runner, session


# Message helper: send text and return the events and updated session.
async def send_message(runner, session, message: str):
    content = types.Content(
        role="user",
        parts=[types.Part.from_text(text=message)],
    )
    events = [
        event
        async for event in runner.run_async(
            user_id=session.user_id,
            session_id=session.id,
            new_message=content,
        )
    ]
    current = await runner.session_service.get_session(
        app_name=session.app_name,
        user_id=session.user_id,
        session_id=session.id,
    )
    return events, current


# Display helper: extract the last final response from an event list.
def final_text(events) -> str:
    for event in reversed(events):
        if event.is_final_response() and event.content:
            return "".join(part.text or "" for part in event.content.parts)
    return ""


# Display helper: shorten structured data so traces stay readable.
def _compact(value, limit: int = 700) -> str:
    rendered = pformat(value, width=88, sort_dicts=False)
    return rendered if len(rendered) <= limit else rendered[:limit] + " ..."


# Display helper: show agent activity, tool results, and state changes as a timeline.
def show_event_trace(events):
    """Display a semantic ADK timeline, including explicit agent handoffs.

    Transfer events are special: the generated function response is ``None`` because
    ADK records the destination in ``event.actions.transfer_to_agent``. Collapse that
    call/result pair into one handoff so a multi-agent run is not mistaken for a
    single-agent tool call.
    """
    steps = []
    shown_handoffs = set()

    agent_path = []
    for event in events:
        author = str(getattr(event, "author", None) or "agent")
        if not agent_path or agent_path[-1] != author:
            agent_path.append(author)

    # Display helper: add one labeled card to the execution trace.
    def add_step(kind, title, author, body, color):
        number = len(steps) + 1
        steps.append(f"""
        <div class="trace-step" style="--trace-color: {color}">
          <div class="trace-marker">{number}</div>
          <div class="trace-card">
            <div class="trace-meta">{escape(kind)} · {escape(str(author))}</div>
            <div class="trace-title">{escape(str(title))}</div>
            {body}
          </div>
        </div>""")

    # Display helper: format a labeled value for a trace card.
    def data_block(label, value):
        return (
            f'<div class="trace-label">{escape(label)}</div>'
            f'<pre>{escape(_compact(value))}</pre>'
        )

    for event in events:
        author = str(getattr(event, "author", None) or "agent")
        calls = list(event.get_function_calls())
        results = list(event.get_function_responses())
        actions = getattr(event, "actions", None)
        transfer_target = getattr(actions, "transfer_to_agent", None)

        # The call event arrives before ADK puts the destination on the response event.
        # Reading the call arguments lets the handoff appear in its true timeline slot.
        if not transfer_target:
            for call in calls:
                if call.name == "transfer_to_agent":
                    transfer_target = (call.args or {}).get("agent_name")
                    if transfer_target:
                        break

        if transfer_target:
            transfer_target = str(transfer_target)
            handoff = (author, transfer_target)
            if handoff not in shown_handoffs:
                shown_handoffs.add(handoff)
                detail = (
                    '<div class="trace-response">Control passed to the selected '
                    f'specialist: <code>{escape(transfer_target)}</code>.</div>'
                )
                add_step(
                    "AGENT HANDOFF",
                    f"{author} → {transfer_target}",
                    author,
                    detail,
                    "#0891b2",
                )

        for call in calls:
            if call.name != "transfer_to_agent":
                add_step("TOOL CALL", call.name, author, data_block("Arguments", call.args), "#2563eb")
        for result in results:
            if result.name != "transfer_to_agent":
                add_step("TOOL RESULT", result.name, author, data_block("Returned", result.response), "#7c3aed")
        state_delta = getattr(actions, "state_delta", None)
        if state_delta:
            add_step("STATE UPDATE", "Session state changed", author, data_block("Changed values", state_delta), "#d97706")
        if event.is_final_response() and event.content:
            text = "".join(part.text or "" for part in event.content.parts).strip()
            if text:
                response = f'<div class="trace-response">{escape(text)}</div>'
                add_step("FINAL RESPONSE", "Reply to customer", author, response, "#059669")

    count = len(steps)
    body = "".join(steps) or '<div class="trace-empty">No visible events in this run.</div>'
    styles = """
    <style>
      .event-trace { max-width: 900px; margin: 12px 0; color: var(--jp-content-font-color1, #172033); font-family: var(--jp-ui-font-family, system-ui, sans-serif); }
      .trace-heading { font-size: 14px; font-weight: 700; margin-bottom: 4px; }
      .trace-path { color: #475569; font-size: 12px; margin-bottom: 12px; }
      .trace-path code { font-size: 11px; }
      .trace-count, .trace-label { color: #64748b; font-weight: 600; }
      .trace-step { position: relative; display: grid; grid-template-columns: 32px 1fr; gap: 10px; padding-bottom: 12px; }
      .trace-step:not(:last-child)::before { content: ""; position: absolute; left: 14px; top: 30px; bottom: -2px; width: 2px; background: #dbe3ee; }
      .trace-marker { z-index: 1; width: 30px; height: 30px; border-radius: 50%; background: var(--trace-color); color: white; display: grid; place-items: center; font-size: 13px; font-weight: 700; }
      .trace-card { border: 1px solid #dbe3ee; border-left: 4px solid var(--trace-color); border-radius: 8px; padding: 10px 14px 12px; background: var(--jp-layout-color1, #fff); }
      .trace-meta { color: var(--trace-color); font-size: 11px; font-weight: 800; letter-spacing: .06em; }
      .trace-title { font-size: 15px; font-weight: 700; margin: 2px 0 9px; }
      .trace-label { font-size: 11px; text-transform: uppercase; margin-bottom: 4px; }
      .trace-card pre { margin: 0; padding: 9px 11px; border-radius: 6px; background: var(--jp-cell-editor-background, #f6f8fa); white-space: pre-wrap; overflow-wrap: anywhere; font-size: 12px; line-height: 1.45; }
      .trace-response { white-space: pre-wrap; line-height: 1.55; }
      .trace-empty { color: #64748b; padding: 12px; border: 1px dashed #cbd5e1; border-radius: 8px; }
    </style>
    """
    heading = f'Execution trace <span class="trace-count">· {count} {"step" if count == 1 else "steps"}</span>'
    path = " → ".join(f"<code>{escape(agent)}</code>" for agent in agent_path)
    path_html = f'<div class="trace-path">Agent path: {path}</div>' if path else ""
    display(HTML(styles + f'<div class="event-trace"><div class="trace-heading">{heading}</div>{path_html}{body}</div>'))

In [4]:
# Live example: ask the agent a general question that needs no policy lookup.
general_query = "What does an insurance deductible mean?"

if RUN_LLM_EXAMPLES:
    general_runner, general_session = await start_agent_session(
        general_claims_agent, "claims_concepts_demo"
    )
    general_events, general_session = await send_message(
        general_runner, general_session, general_query
    )
    print("CUSTOMER:", general_query)
    show_event_trace(general_events)
else:
    print("Live example skipped. Add an API key or set RUN_LLM_EXAMPLES=true to run it.")

CUSTOMER: What does an insurance deductible mean?


**Key idea:** Instructions guide the agent's behavior. A tool gives it access to policy facts.

![From Model to Agent to Tool](Images/02_model_agent_tool.png)

## 2. Give the agent a tool

A **tool** is a function the agent can call. Here, `lookup_policy` reads a record by policy number. Its docstring explains its purpose, and its type hints describe its inputs and output.

Call it directly first. **Look for:** a `found` result for a known policy and `not_found` for an unknown one.

In [5]:
# Policy tool: find a policy by its exact number and return the lookup result.
def lookup_policy(policy_number: str) -> dict:
    """Look up an auto policy by its exact policy number.

    Args:
        policy_number: Identifier in the form "POL-######", for example "POL-100234".

    Returns:
        A found result containing the policy, or a not_found result.
    """
    policy = POLICIES.get(policy_number)
    if policy is None:
        return {"status": "not_found", "policy_number": policy_number}
    return {"status": "found", "policy_number": policy_number, "policy": policy}


# Direct example: compare successful and unsuccessful lookups before involving an agent.
# print(_compact(lookup_policy("POL-100234")))
# print(lookup_policy("POL-999999"))

Now add the function to the agent's `tools` list. The model chooses when to call it; Python performs the lookup.

**Look for:** tool call → returned policy → reply based on that policy.

In [6]:
# Policy agent: use lookup_policy to answer questions about a specific policy.
policy_agent = Agent(
    name="policy_agent",
    model=MODEL,
    description="Answers questions about a specific auto policy.",
    instruction=(
        "For every policy-specific fact, call lookup_policy. "
        "Explain only the returned facts in plain language. "
        "If the policy is not found, ask the customer to verify the number."
    ),
    tools=[lookup_policy],
)

policy_query = "What is the collision deductible for policy POL-100234?"

if RUN_LLM_EXAMPLES:
    policy_runner, policy_session = await start_agent_session(
        policy_agent, "policy_lookup_demo"
    )
    policy_events, policy_session = await send_message(
        policy_runner, policy_session, policy_query
    )
    print("CUSTOMER:", policy_query)
    print()
    show_event_trace(policy_events)
else:
    print("Live tool-call trace skipped. The deterministic lookup above still ran.")

./.venv/lib/python3.12/site-packages/google/adk/models/llm_request.py:298: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


CUSTOMER: What is the collision deductible for policy POL-100234?



## 3. Use several tools in one agent

One request may need several sources. Here, `lookup_policy` supplies the deductible, and `lookup_part_price` supplies catalog cost and labor hours.

**Key idea:** Both tools serve the same `policy_support_agent`. Adding a tool adds a capability.

![One Agent, Multiple Tools](Images/03_one_agent_multiple_tools.png)

In [7]:
# Pricing tool: return the catalog cost and labor hours for one vehicle part.
def lookup_part_price(part_name: str) -> dict:
    """Look up the base cost and labor hours for one supported vehicle part.

    Args:
        part_name: A catalog key such as "front_bumper", "rear_bumper",
            "headlight", "hood", or "windshield".

    Returns:
        The catalog entry, or a not_found result listing a few valid examples.
    """
    price = PARTS_PRICING.get(part_name)
    if price is None:
        return {
            "status": "not_found",
            "part_name": part_name,
            "examples": ["front_bumper", "rear_bumper", "headlight", "hood"],
        }
    return {"status": "found", "part_name": part_name, "pricing": price}


# Support agent: combine policy and parts-catalog facts with two tools.
policy_support_agent = Agent(
    name="policy_support_agent",
    model=MODEL,
    description="Looks up policy facts and basic repair-catalog facts.",
    instruction=(
        "Use lookup_policy for policy facts and lookup_part_price for catalog facts. "
        "For a compound question, call every tool needed before answering. "
        "Do not turn a catalog price into a claim payout or final repair estimate."
    ),
    tools=[lookup_policy, lookup_part_price],
)

In [8]:
# Combined request: ask one agent for facts from both data sources.
multi_tool_query = (
    "For POL-100234, what is my deductible, and what base part cost and labor "
    "hours are listed for a front_bumper?"
)

if RUN_LLM_EXAMPLES:
    support_runner, support_session = await start_agent_session(
        policy_support_agent, "multi_tool_demo"
    )
    support_events, support_session = await send_message(
        support_runner, support_session, multi_tool_query
    )
    print("CUSTOMER:", multi_tool_query)
    show_event_trace(support_events)
else:
    print("Policy result:", _compact(lookup_policy("POL-100234")))
    print("Part result:", lookup_part_price("front_bumper"))

CUSTOMER: For POL-100234, what is my deductible, and what base part cost and labor hours are listed for a front_bumper?


## 4. Carry facts between messages

Helen says, “My policy is POL-100234,” then asks, “Does it cover a collision?” Both messages below use the same session.

| Concept | What it holds in this example |
|---|---|
| **Session** | The ongoing conversation, including its events and state |
| **State** | The selected `policy_number` and `policy` record |
| **Events** | Messages, tool calls, results, and state changes |

ADK supplies `ToolContext` to the tools so they can read and update session state.

![Sessions, State and Events](Images/04_sessions_state_events.png)

In [9]:
from google.adk.tools import ToolContext


# State tool: look up a policy and save the found record for follow-up questions.
def select_policy(policy_number: str, tool_context: ToolContext) -> dict:
    """Look up a policy and save the found record in the current session.

    Args:
        policy_number: Identifier in the form "POL-######".
        tool_context: Supplied by ADK; provides access to session state.
    """
    result = lookup_policy(policy_number)
    if result["status"] == "found":
        tool_context.state["policy_number"] = policy_number
        tool_context.state["policy"] = result["policy"]
    return result


# Coverage tool: check a coverage type using the policy already saved in state.
def check_saved_coverage(incident_type: str, tool_context: ToolContext) -> dict:
    """Check one coverage type on the policy already saved in session state.

    Args:
        incident_type: Either "collision" or "comprehensive".
        tool_context: Supplied by ADK; provides the saved policy.
    """
    if incident_type not in {"collision", "comprehensive"}:
        return {"status": "invalid_incident_type", "allowed": ["collision", "comprehensive"]}
    policy = tool_context.state.get("policy")
    if not policy:
        return {"status": "missing_policy", "reason": "Select a policy first."}
    covered = bool(policy["coverage"].get(incident_type))
    return {
        "status": "covered" if covered else "not_covered",
        "incident_type": incident_type,
        "deductible": policy["coverage"]["deductible"],
    }


# Stateful agent: save a selected policy and reuse it for coverage questions.
stateful_policy_agent = Agent(
    name="stateful_policy_agent",
    model=MODEL,
    description="Selects a policy and answers follow-up coverage questions.",
    instruction="""
When the customer supplies a policy number, call select_policy; that necessary
lookup also saves the policy in session state. For a later collision or
comprehensive question with no repeated number, call check_saved_coverage.
Never ask for a policy number that is already available in the session.
""",
    tools=[select_policy, check_saved_coverage],
)

In [10]:
# Two-turn example: provide a policy number, then ask about coverage in the same session.
first_message = (
    input("First message: ")
    if INTERACTIVE_INPUTS
    else "My policy number is POL-100234."
)
follow_up_message = (
    input("Follow-up message: ")
    if INTERACTIVE_INPUTS
    else "Does it cover a collision?"
)

if RUN_LLM_EXAMPLES:
    state_runner, state_session = await start_agent_session(
        stateful_policy_agent, "policy_state_demo"
    )

    print("CUSTOMER:", first_message)
    first_events, state_session = await send_message(
        state_runner, state_session, first_message
    )
    show_event_trace(first_events)
    print("SESSION STATE:", _compact(state_session.state))

    print("\nCUSTOMER:", follow_up_message)
    follow_up_events, state_session = await send_message(
        state_runner, state_session, follow_up_message
    )
    show_event_trace(follow_up_events)
    print("SESSION STATE:", _compact(state_session.state))
else:
    print("Live two-turn example skipped. Set RUN_LLM_EXAMPLES=true to run it.")

CUSTOMER: My policy number is POL-100234.


SESSION STATE: {'policy_number': 'POL-100234',
 'policy': {'policyholder_id': 'CUST-5521',
            'policyholder_name': 'Helen Parr',
            'vin': '1HGCM82633A004352',
            'listed_drivers': ['Helen Parr', 'Bob Parr'],
            'coverage': {'collision': True,
                         'comprehensive': True,
                         'deductible': 500,
                         'limits': 25000},
            'effective_date': '2025-01-01',
            'expiration_date': '2027-01-01',
            'exclusions': ['racing', 'commercial_use', 'unlisted_driver']}}

CUSTOMER: Does it cover a collision?


SESSION STATE: {'policy_number': 'POL-100234',
 'policy': {'policyholder_id': 'CUST-5521',
            'policyholder_name': 'Helen Parr',
            'vin': '1HGCM82633A004352',
            'listed_drivers': ['Helen Parr', 'Bob Parr'],
            'coverage': {'collision': True,
                         'comprehensive': True,
                         'deductible': 500,
                         'limits': 25000},
            'effective_date': '2025-01-01',
            'expiration_date': '2027-01-01',
            'exclusions': ['racing', 'commercial_use', 'unlisted_driver']}}


**Look for:** `select_policy` saves the record on the first turn; `check_saved_coverage` reads it on the second.

**Key idea:** Save useful facts during a lookup so later tools can reuse them.

## 5. Route a request to a specialist

A **router** chooses the specialist whose job matches the request. Each specialist has its own instructions and available tools.

```text
Customer request
      ├── Policy question specialist
      └── Claim intake specialist
```

The router's `sub_agents` list provides the destinations. Their `description` fields explain when to choose each one.

![Routing to One Specialist](Images/05_agent_routing_specialists.png)

In [11]:
# Policy specialist: answer policy questions using the lookup tool.
policy_question_specialist = Agent(
    name="policy_question_specialist",
    model=MODEL,
    description="Handles questions about an identified policy, deductible, or coverage.",
    instruction="Answer only policy questions. Use lookup_policy for every policy fact.",
    tools=[lookup_policy],
)

# Intake specialist: explain what information a customer needs to start a claim.
claim_intake_specialist = Agent(
    name="claim_intake_specialist",
    model=MODEL,
    description="Helps a customer begin a new auto claim and lists the information needed.",
    instruction=(
        "Explain the basic information needed to start a claim in a calm tone. "
        "Do not assess coverage, risk, repair cost, or approval."
    ),
)

# Router agent: hand the request to the specialist whose description matches it.
claims_router = Agent(
    name="claims_router",
    model=MODEL,
    description="Routes an insurance request to the appropriate specialist.",
    instruction=(
        "Delegate the request to exactly one specialist whose description matches it. "
        "Do not answer the specialist's question yourself."
    ),
    sub_agents=[policy_question_specialist, claim_intake_specialist],
)

In [12]:
# Routing example: follow a new-claim request from the router to the intake specialist.
routing_query = "I was just in an accident and need to start a new claim."

if RUN_LLM_EXAMPLES:
    routing_runner, routing_session = await start_agent_session(
        claims_router, "claims_routing_demo"
    )
    routing_events, routing_session = await send_message(
        routing_runner, routing_session, routing_query
    )
    print("CUSTOMER:", routing_query)
    show_event_trace(routing_events)
else:
    print("Live routing example skipped. Expected destination: claim_intake_specialist.")

App "claims_routing_demo" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


CUSTOMER: I was just in an accident and need to start a new claim.


**Look for:** **AGENT HANDOFF** and the **Agent path** in the trace. For this request, `claim_intake_specialist` should take over and write the reply.

**Key idea:** Routing chooses who handles the request. The final response's author shows which specialist answered.

## 6. Run required stages with a workflow

| Need | Pattern | Who chooses the path? |
|---|---|---|
| Choose a matching specialist | Routing | The coordinating model |
| Run required stages in order | Workflow | The application's graph |

The graph below runs **select policy → check saved coverage**. Shared state carries the policy between the two agents.

`mode="single_turn"` lets each agent finish its stage without another customer conversation. It can still make several tool calls.

![Routing vs Workflow](Images/06_routing_vs_workflow.png)

In [13]:
from google.adk.workflow import START, Workflow


# First workflow agent: select and save the policy needed by the next stage.
workflow_policy_agent = Agent(
    name="workflow_policy_agent",
    model=MODEL,
    mode="single_turn",
    description="Selects the policy required by a coverage request.",
    instruction=(
        "Extract the policy number from the request and call select_policy. "
        "Then return a short note that preserves the requested incident type."
    ),
    tools=[select_policy],
)

# Second workflow agent: check coverage using the policy saved by the first stage.
workflow_coverage_agent = Agent(
    name="workflow_coverage_agent",
    model=MODEL,
    mode="single_turn",
    description="Checks the requested coverage using the policy stored by the prior stage.",
    instruction=(
        "Call check_saved_coverage for the incident type in the incoming note, "
        "then explain the returned status and deductible."
    ),
    tools=[check_saved_coverage],
)

# Workflow: require policy selection before the coverage check.
policy_coverage_workflow = Workflow(
    name="policy_coverage_workflow",
    edges=[(START, workflow_policy_agent, workflow_coverage_agent)],
)

print("WORKFLOW:", " → ".join(node.name for node in policy_coverage_workflow.graph.nodes))

WORKFLOW: __START__ → workflow_policy_agent → workflow_coverage_agent


In [14]:
# Workflow example: run both required stages and inspect their shared state.
workflow_query = "For policy POL-100234, check whether collision is covered."

if RUN_LLM_EXAMPLES:
    workflow_runner = InMemoryRunner(
        node=policy_coverage_workflow,
        app_name="coverage_workflow_demo",
    )
    workflow_session = await workflow_runner.session_service.create_session(
        app_name="coverage_workflow_demo",
        user_id="workshop-user",
    )
    workflow_events, workflow_session = await send_message(
        workflow_runner, workflow_session, workflow_query
    )
    print("CUSTOMER:", workflow_query)
    show_event_trace(workflow_events)
    print("FINAL STATE:", _compact(workflow_session.state))
else:
    print("Live workflow skipped. Its enforced order is shown above.")

CUSTOMER: For policy POL-100234, check whether collision is covered.


FINAL STATE: {'policy_number': 'POL-100234',
 'policy': {'policyholder_id': 'CUST-5521',
            'policyholder_name': 'Helen Parr',
            'vin': '1HGCM82633A004352',
            'listed_drivers': ['Helen Parr', 'Bob Parr'],
            'coverage': {'collision': True,
                         'comprehensive': True,
                         'deductible': 500,
                         'limits': 25000},
            'effective_date': '2025-01-01',
            'expiration_date': '2027-01-01',
            'exclusions': ['racing', 'commercial_use', 'unlisted_driver']}}


## Apply these concepts next

In [Notebook 2](02_Insurance_Claims_Agent.ipynb), the same building blocks process a complete claim:

```text
START → Intake → Assessment → Escalation → Reply
```

Tools save results in shared state, specialists handle each stage, and events let you follow the claim from customer input to outcome.

## Key takeaways

- **Models** generate responses; **agents** give them a job and capabilities.
- **Tools** provide facts or actions. One agent can use several tools.
- **Sessions** hold the conversation and its state.
- **State** carries structured facts; **events** record what happened.
- **Routing** chooses a specialist for a request.
- **Workflows** control the order of required stages.